In [3]:
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = "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"
)

completion = client.chat.completions.create(
  model="nvidia/llama-3.3-nemotron-super-49b-v1",
  messages=[{"role":"system","content":"detailed thinking on"}],
  temperature=0.0,
  top_p=0.95,
  max_tokens=4096,
  frequency_penalty=0,
  presence_penalty=0,
  stream=True
)

for chunk in completion:
  if chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")

<think>
Okay, so I need to figure out how to solve this problem: "Find all real solutions to the equation sin(x) + cos(x) = √2 sin(x + π/4)." Hmm, let's start by understanding what's being asked here. The equation involves sine and cosine functions, and we need to find all real x that satisfy it. 

First, I remember that there are some trigonometric identities that might help simplify this equation. The right side has sin(x + π/4), which makes me think of the sine addition formula. The left side is sin(x) + cos(x), which also reminds me of another identity. Maybe I can express both sides in terms of a single trigonometric function or use some identity to combine them.

Let me recall the sine addition formula: sin(a + b) = sin(a)cos(b) + cos(a)sin(b). If I apply that to the right side, sin(x + π/4) would be sin(x)cos(π/4) + cos(x)sin(π/4). Since cos(π/4) and sin(π/4) are both √2/2, that simplifies to (√2/2)(sin(x) + cos(x)). So the right side of the equation becomes √2 * (√2/2)(sin(x) +

In [22]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import faiss
import numpy as np

# 1. Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")  # Fast and efficient

# 2. Load generation model (choose a light one for local inference)
generator = pipeline("text2text-generation", model="google/flan-t5-base")

# 3. Knowledge base (you can replace this with any document list)
documents = [
    "The Eiffel Tower is located in Paris.",
    "Python is a programming language.",
    "The Earth revolves around the Sun.",
    "Tesla was founded by Elon Musk.",
    "Water boils at 100 degrees Celsius."
]

# 4. Embed and index documents
doc_embeddings = embedder.encode(documents)
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

# 5. Retrieval function
def retrieve(query, k=2):
    query_embedding = embedder.encode([query])
    D, I = index.search(np.array(query_embedding), k)
    return [documents[i] for i in I[0]]

# 6. Generation function
def generate_answer(query):
    context = retrieve(query)
    context_text = " ".join(context)
    prompt = f"Context: {context_text}\nQuestion: {query}\nAnswer:"
    result = generator(prompt, max_length=100, do_sample=False)
    return result[0]['generated_text']

# 7. Test query
query = "Who started Tesla?"
answer = generate_answer(query)
print("Answer:", answer)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


Answer: Elon Musk


In [ ]:
import streamlit as st
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.vectorstores import FAISS
import pickle
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

1880484


In [ ]:
import json
import numpy as np
import faiss
from datetime import datetime
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# --- Step 1: Load Transactions ---
with open(r"data/transactions.json", "r") as f:
    transactions = json.load(f)

# --- Step 2: Convert to Searchable Text ---
def tx_to_text(tx):
    try:
        date = datetime.fromisoformat(tx.get("updated_timestamp")).strftime("%B %d, %Y")
    except:
        date = tx.get("updated_timestamp", "unknown date")
    amount = tx.get("amount", 0)
    place = tx.get("place_name") or tx.get("city") or "an unknown location"
    desc = tx.get("transaction_description", "").strip()
    cat = tx.get("category", "Uncategorized")
    verb = "spent" if amount < 0 else "received"
    return f"On {date}, you {verb} €{abs(amount):.2f} at {place} for {cat}. Description: {desc}"

documents = [tx_to_text(tx) for tx in transactions]

# --- Step 3: Embed with Sentence Transformers ---
print("Encoding documents...")
model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = model.encode(documents)

# --- Step 4: Store in FAISS ---
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings))
id_map = {i: doc for i, doc in enumerate(documents)}

Encoding documents...
Encoding documents...


In [29]:
import pickle

# Save FAISS index
faiss.write_index(index, "faiss_transactions.index")

# Save the document ID mapping
with open("doc_id_map.pkl", "wb") as f:
    pickle.dump(id_map, f)